# Test notebook


In [4]:
%load_ext autoreload
%autoreload 1

In [2]:
import sys
import tensorflow as tf
print("Python version:", sys.version)
print("TensorFlow version:", tf.__version__)

2024-08-01 22:01:54.929589: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Python version: 3.11.9 | packaged by conda-forge | (main, Apr 19 2024, 18:36:13) [GCC 12.3.0]
TensorFlow version: 2.16.1


In [ ]:
from vision_models.utils import VisionUtils

vutil = VisionUtils()    
vutil.set_seed()
vutil.print_python_version()
vutil.print_tf_version()
vutil.print_tf_gpu_support()

In [5]:
from vision_models import constants
from vision_models.imageloader import ImageLoader
from vision_models.utils import VisionUtils


vutil1 = VisionUtils()    

loader = ImageLoader(label_coordinates_csv=constants.TRAIN_LABEL_CORD_PATH, labels_csv=constants.TRAIN_LABEL_PATH, 
                              image_dir=constants.TRAIN_DATA_PATH, roi_size=(224, 224), batch_size=1)

#loader._analyze_splits()


In [2]:
val_dataset = loader.load_data("val")

Dataset is created, setting batch size
Batching dataset to : 1
Dataset created, you can now iterate over the dataset


2024-08-03 19:33:16.535473: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:282] failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


In [3]:
for img, labels in val_dataset.take(1):
    print(img.shape)
    print(labels.shape)
    print(labels)

****************************************************************************************************
Going to generate feature for study_id: 4283570761, series_id: 2708429184, condition: Right Neural Foraminal Narrowing, level: L4/L5
Preprocessing images
Reading images from /opt/dataset/train_images//4283570761/2708429184
Number of images in series: 18
Padding tensor to 192 images
Feature tensor generated, size: (192, 224, 224, 3), now generating label
Label generated
One hot vector generated: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
    study_id   series_id  instance_number                         condition  \
0    4003253   702807833                8             Spinal Canal Stenosis   
1    4003253   702807833                8             Spinal Canal Stenosis   
2    4003253   702807833                8             Spinal Canal Stenosis   
3    4003253   702807833                8             Spina

2024-08-03 19:33:20.778213: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [ ]:
from vision_models.densenetmodel import DenseNetVisionModel
import tensorflow as tf

# Make sure input_shape is fully defined
input_shape = (224,224,3)

# Create an instance of the model
model = DenseNetVisionModel(num_classes=25, input_shape=input_shape, weights='imagenet')

# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Build the model with a sample input
sample_input = tf.keras.Input(shape=input_shape)
model(sample_input)

In [ ]:
from vision_models.densenetmodel import ModelTrainer

# Use the original model
trainer = ModelTrainer(model)

history = trainer.train(train_dataset, val_dataset, epochs=5)

In [1]:
import google.cloud.storage
print(google.cloud.storage.__file__)

/opt/conda/envs/vision/lib/python3.11/site-packages/google/cloud/storage/__init__.py


In [5]:
ls -lh /tmp/model.h5


-rw-r--r-- 1 dhartipatelseagraves dhartipatelseagraves 29M Aug 12 21:14 /tmp/model.h5


In [23]:
import os

model_path = 'vision_models_stage_1_Densenet_best_model.weights_08112024_1414.h5'
size = os.path.getsize(model_path)
print(f"File size: {size} bytes")


File size: 29508160 bytes


In [26]:
h5ls 'vision_models_stage_1_Densenet_best_model.weights_08112024_1414.h5'


SyntaxError: invalid syntax (3790450149.py, line 1)

In [27]:
import h5py

# model_path = '/tmp/model.h5'
model_path = 'vision_models_stage_1_Densenet_best_model.weights_08112024_1414.h5'
with h5py.File(model_path, 'r') as f:
    print(f.keys())  # Lists all the top-level groups in the HDF5 file


<KeysViewHDF5 ['base_model', 'global_average_layer', 'layers', 'optimizer', 'vars']>


In [29]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from google.cloud import storage
from keras.models import load_model
import vision_models.constants as constants
from vision_models.densenetmodel import DenseNetVisionModel  # Import your model definition
from vision_models.dataset import Dataset
from vision_models.constants import TEST_DATA_PATH, TRAIN_LABEL_PATH, IMAGE_SIZE_HEIGHT, IMAGE_SIZE_WIDTH, TRAIN, TEST, VAL, DISEASE_THRESHOLD

# Initialize Dataset
dataset = Dataset(batch_size=constants.BATCH_SIZE)

# Extract labels from the dataset
labels = dataset.label_list

expected_input_shape = (200, 224, 224, 3)
input_layer = tf.keras.layers.Input(shape=(200, 224, 224, 3))


model = DenseNetVisionModel(num_classes=len(labels), input_shape=expected_input_shape, weights=None)
model.summary()


####################################################################################################
Dataset splits sizes: {'train': 29214, 'val': 9739, 'test': 9739}
Input shape received to the init method: (200, 224, 224, 3)
Input shape for base model: (224, 224, 3)


ValueError: Undefined shapes are not supported.

In [34]:
import tensorflow as tf
from keras import layers, models

def create_simple_model(input_shape):
    inputs = tf.keras.Input(shape=input_shape)
    x = layers.Conv2D(32, (3, 3), activation='relu')(inputs)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Flatten()(x)
    outputs = layers.Dense(10, activation='softmax')(x)
    
    model = models.Model(inputs, outputs)
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    
    return model

# Test with a simplified input shape
simple_input_shape = (224, 224, 3)  # or (200, 224, 224, 3) depending on your needs

try:
    model = create_simple_model(simple_input_shape)
    model.summary()  # This should work without errors
except ValueError as e:
    print(f"Error in simple model summary: {e}")


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_10 (InputLayer)     │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 394272)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 10)             │     3,942,730 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,943,626 (15.04 MB)

 Trainable params: 3,943,626 (15.04 MB)

 Non-trainable params: 0 (0.00 B)

In [36]:
import h5py

model_path = 'vision_models_stage_1_Densenet_best_model.weights_08112024_1414.h5'

with h5py.File(model_path, 'r') as f:
    # List all groups
    print("Keys: %s" % f.keys())
    for key in f.keys():
        print(f"{key}: {list(f[key].keys())}")


Keys: <KeysViewHDF5 ['base_model', 'global_average_layer', 'layers', 'optimizer', 'vars']>
base_model: ['layers', 'vars']
global_average_layer: ['vars']
layers: ['dense']
optimizer: ['vars']
vars: []


In [37]:
import numpy as np

dummy_input = np.random.random((1, 7, 7, 1024))  # Example shape
pooling_layer = tf.keras.layers.GlobalAveragePooling2D()
output = pooling_layer(dummy_input)
print(output.shape)  # This should give you the pooled output shape


(1, 1024)


In [32]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from keras.models import load_model
import vision_models.constants as constants
from vision_models.densenetmodel import DenseNetVisionModel
from vision_models.dataset import Dataset

# Initialize Dataset
dataset = Dataset(batch_size=constants.BATCH_SIZE)

# Extract labels from the dataset
labels = dataset.label_list

# Test with a simplified input shape
expected_input_shape = (224, 224, 3)

try:
    # Initialize the model with the simpler input shape
    model = DenseNetVisionModel(num_classes=len(labels), input_shape=expected_input_shape, weights=None)
    model.summary()  # This should now work without errors
except ValueError as e:
    print(f"Error in model summary: {e}")


####################################################################################################
Dataset splits sizes: {'train': 29214, 'val': 9739, 'test': 9739}
Input shape received to the init method: (200, 224, 224, 3)
Input shape for base model: (224, 224, 3)
Error in model summary: Undefined shapes are not supported.


In [22]:
from tensorflow.python.lib.io import file_io
model_file = file_io.FileIO('gs://models_output_234324/vision_models/stage_1/Densenet/best_model.weights_08112024_1414.h5', mode='rb')

temp_model_location = '/tmp/temp_model.h5'
temp_model_file = open(temp_model_location, 'wb')
temp_model_file.write(model_file.read())
temp_model_file.close()
model_file.close()

model = tf.keras.models.load_model(temp_model_location)

ValueError: No model config found in the file at /tmp/temp_model.h5.